# A1.8 · Malicious code execution

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.7 · Identity spoofing and impersonation](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**.

| | |
|---|---|
| Tools used | Falco, gVisor, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Execute model-authored code and enumerate what the process could touch.

**Why a security engineer needs it.** Model-authored code runs with the runtime's privileges — reaching the filesystem, the network and any credential in the environment. The control it builds is: sandboxed execution (A3.2) and egress control (A3.3).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Asking a model to write code is safe. Running the code it wrote is the part that is not, and most agent frameworks ship the second one enabled with the same process privileges as the framework itself.

> **At CyberTravels.** The Coding Agent writes a patch and the runtime executes it. On Alex's laptop that process can read `~/.aws`, the HR folder and the roadmap directory, because nothing said otherwise. R6.

## 2 · The framework

```
   model ---> "here is a script that does it" ---> agent runtime
                                                        |
                                            exec() on the host
                                                        v
                                        whatever the PROCESS can reach:
                                        files . network . credentials . socket

   writing the code is safe. running it is the part that is not.
```

**OWASP T11 — Unexpected RCE and Code Attacks. LLM05 — Improper Output Handling.**

Many useful agents write code and run it — that is what makes a data-analysis
agent or a coding agent worth having. The **agent_runtime** component executes
text the **model** produced, on a host, in a process.

The risk is not exotic. Model-authored code is just code, and it runs with
whatever the process has: the filesystem it can see, the network it can reach,
and every credential in its environment. There is no privilege boundary between
"the code the agent wrote to reformat a CSV" and "the code that reads
`~/.aws/credentials`", because both are strings passed to the same interpreter.

Two paths lead here, and only one involves an attacker:

**Steered.** An injection from A1.3 tells the agent to write particular code.
The runtime executes it because executing code is its job.

**Unsteered.** Nobody attacked anything. The agent wrote something plausible and
wrong — a cleanup routine with a path variable that resolves higher than
intended — and the blast radius was decided by the environment, not by intent.

That second path is worth sitting with. Most teams model this as an attack. In
practice the first incident is usually an ordinary bug with production
credentials in scope, which is why the control in A3.2 is about what the process
can *reach*, not about what the model can be persuaded to *write*.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

What the executing process can reach, enumerated rather than assumed. Nothing below actually touches your machine — the environment is a fixture, so the lesson runs anywhere.

## 4 · The check, as a skill

CyberTravels' Coding Agent runs code it wrote. The skill enumerates what that process reaches — environment, filesystem, network — on an ordinary task first, because the ordinary task is the more persuasive half of the finding.

### The skill — [`skills/threats/generated-code-reach-enumerator/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/generated-code-reach-enumerator/SKILL.md)

```yaml
name: generated-code-reach-enumerator
description: >-
  Enumerate what model-authored code can read, write and connect to when it
  executes — on an ordinary task and on a steered one — including process
  environment, filesystem and cloud metadata. Use when an agent runs code it
  wrote, or when sizing the runtime that code should execute in.
allowed-tools: Read, Grep, Glob, Bash
```

# The reach is the same whether the code was steered or not

An agent that executes its own code has the process's reach, not the task's.
The interesting measurement is that an **ordinary, unattacked** task already
touches everything the process can see; steering only changes what it does with
that reach, not how much of it there is.

## When to use this

Any agent with a code-execution tool, a notebook runner, a build step it
authors, or a shell. Run it before choosing a sandbox, because the output is
the requirement list for one.

## Procedure

**1 — Inventory the process environment.** Every variable visible to the
executing process. Credentials in the environment are reachable by any line of
code, and the agent did not have to look for them.

**2 — Inventory filesystem reach.** What the process can open, not what the
task needs. Include the agent's own configuration, adjacent workspaces, and any
key material mounted for another purpose.

**3 — Inventory network reach.** Resolve and attempt each destination the
process can open. The cloud metadata address is the one that turns a code
execution into a credential theft; test it explicitly.

**4 — Run the benign task and record what it touched.** This is the number that
changes the conversation: an ordinary task with no adversary reaching a private
key is a design fact, not an incident.

**5 — Run the steered task and diff.** The difference between the two is what
an attacker gains. It is usually smaller than people expect, because the
ordinary run already had everything.

## Output contract

```json
{
  "environment": {"variables": ["str"], "credential_shaped": ["str"]},
  "filesystem": {"readable": ["str"], "sensitive": ["str"]},
  "network": {"reachable": ["str"], "metadata_endpoint": true},
  "benign_run": {"touched": ["str"]},
  "steered_run": {"touched": ["str"], "gain_over_benign": ["str"]},
  "sandbox_requirements": ["str"]
}
```

## Failure modes

- **Measuring the task instead of the process.** The task is a suggestion; the
  process boundary is the control.
- **Skipping the metadata endpoint** because it is not in the code. It does not
  need to be.
- **Reporting only the steered run.** The benign run is the more persuasive
  half of the finding.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/generated-code-reach-enumerator/scripts/generated_code_reach_enumerator.py
SCRIPT = "skills/threats/generated-code-reach-enumerator/scripts/generated_code_reach_enumerator.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Model-authored code is executed against a fixture environment and the reach is enumerated: an ordinary, unattacked task touches every file the process can see including a private key, and steered code reaches the environment credentials and the cloud metadata address.

## Your turn

For one agent that executes code, list what is in its process environment right now. The credentials in that list are the blast radius of the next ordinary bug, not of the next attack.

---

**Next → [A1.9 · Injection through content the agent was asked to read](https://spbreed.github.io/cyber-commons/lessons/A1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*